# 01 · Evaluacion de LLMs con lm-evaluation-harness · base vs instruct

> Demo práctica del módulo de Evaluación. Cargas Qwen3-4B base, lo mides en 3 benchmarks. Luego cargas Qwen3-4B instruct, mides los mismos benchmarks. **Cierras con una tabla comparativa** que ensena el efecto del instruct tuning en numeros reales... y por que esos numeros dependen del protocolo de evaluacion.

## Que aprendemos

1. **lm-evaluation-harness** · la libreria de Eleuther AI que alimenta el Open LLM Leaderboard.
2. **3 benchmarks clasicos** · HellaSwag (sentido comun), ARC-Easy (razonamiento basico), MMLU-ML (conocimiento).
3. **Diferencia base vs instruct** · ¿cuanto sube por hacer instruct tuning?
4. **`acc` vs `acc_norm`** · la metrica canonica de HellaSwag (length-normalized).
5. **Cuantizacion 4-bit en evaluacion** · sirve para fitting en T4 sin afectar el ranking relativo.

## Por que importa

La pregunta natural al terminar el módulo de Fine-tuning es **¿cuanto mejora mi modelo?**. Este módulo te da el lenguaje para responderla con numeros publicos comparables.

Tambien aterriza la critica honesta · los benchmarks **clasicos** (MMLU, HumanEval, ARC-AGI-1) estan saturados en 2026. La frontera real es **MMLU-Pro, GPQA Diamond, SWE-bench Pro, HLE, ARC-AGI-2**. Lo veras en las slides 20 y 26.

## Conecta con

- **Slides de Evaluación** · benchmarks clasicos (9-17), critica (18-19), frontier vs local (20), este notebook (24), benchmarks vigentes 2026 (26).
- **Lab 09 de Fine-tuning** · alli mediste tus adapters de los labs 04 (Milei) y 05 (distillation) con una version minima (ARC con 20 preguntas, toxicidad, juez). Aqui ves la herramienta estandar que lo hace a escala.
- **Labs 04 y 05 de Fine-tuning** · al final tienes la receta para pasar tus adapters por este mismo harness.
- **Módulo de evaluación de LLM apps (clase 6)** · aqui evaluas el MODELO · alli evaluaras tu APP entera.

## Stack

`Qwen3-4B` base + instruct (selector `MODEL_SIZE`: 14B en A100) · `lm-evaluation-harness` (Eleuther AI) · `BitsAndBytesConfig` (NF4 4-bit) · `pandas` (comparativa).

## Hardware

T4 (Colab Free, 16GB VRAM) vale con 4-bit. Con `LIMIT = 500` (por defecto) el notebook entero tarda **~35 min** en T4 (ejecucion de referencia, oct-2026). Con `LIMIT = None` (HellaSwag completo, ~40k pasadas por modelo) son horas en T4: dejalo para A100/H100.


---

## Setup

`lm-evaluation-harness` es el framework de referencia. Lo instalamos **una sola vez** desde PyPI (`lm-eval[hf]`): trae el backend de HuggingFace y no toca las librerias preinstaladas de Colab mas de lo necesario. **Tambien** existe `lighteval` (HuggingFace) que cubre lo mismo con API mas moderna · la veremos en la celda final.


In [1]:
!pip install -q "lm-eval[hf]" bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 4.3 MB/s eta 0:00:00


  Preparing metadata (setup.py) ... done


  Preparing metadata (setup.py) ... done


  Preparing metadata (setup.py) ... done


   ━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/43.1 MB 183.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 43.1/43.1 MB 212.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 43.1/43.1 MB 212.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━ 6.3/9.1 MB 200.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/129.6 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.0 MB/s eta 0:00:00


### Login HuggingFace

`Qwen3-4B-Base` y `Qwen3-4B` son **open weights (Apache 2.0)**: no hay que aceptar ninguna licencia gated. Solo necesitas tu [token de HF](https://huggingface.co/settings/tokens) para el login (descargas mas rapidas):


In [3]:
# Login a HuggingFace (token desde huggingface.co/settings/tokens)
from huggingface_hub import notebook_login
notebook_login()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


---

## Configuracion de evaluacion (reutilizable)

Definimos todo lo que NO cambia entre los 2 modelos: BitsAndBytesConfig + lista de benchmarks + funcion para correr eval. Asi solo cambia `MODEL_ID` entre las 2 partes.


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Quant config canonica · NF4 + double quant (paper QLoRA).
# Compute en bf16 si la GPU lo soporta de forma nativa (Ampere o posterior: A100/L4/H100).
# La T4 (Turing, capacidad 7.5) solo lo emula y va mucho mas lenta -> fp16.
# (Ojo: torch.cuda.is_bf16_supported() devuelve True en T4 porque cuenta la emulacion.)
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
print("Compute dtype:", COMPUTE_DTYPE)

# Modelo a evaluar: base vs instruct del MISMO tamano. Cambia MODEL_SIZE y listo.
MODEL_SIZE = "mini"   # "mini" (T4, 4-bit) · "big" (A100/H100)
MODELS = {
    "mini": ("Qwen/Qwen3-4B-Base",  "Qwen/Qwen3-4B"),    # ~4B en 4-bit cabe en T4 (Qwen3, 2025)
    "big":  ("Qwen/Qwen3-14B-Base", "Qwen/Qwen3-14B"),   # 14B -> A100/H100
}
MODEL_ID_BASE, MODEL_ID_INSTRUCT = MODELS[MODEL_SIZE]
print("Base:", MODEL_ID_BASE, "| Instruct:", MODEL_ID_INSTRUCT)

# Benchmarks que usaremos
TASKS_BASIC = ["hellaswag", "arc_easy"]
TASK_MMLU = "mmlu_flan_n_shot_loglikelihood_machine_learning"
TASK_MMLU_COT = "mmlu_flan_cot_zeroshot_machine_learning"

# Ejemplos por benchmark. HellaSwag completo son 10.042 preguntas x 4 finales = ~40k forward passes:
# en una T4 son HORAS. Con LIMIT = 500 (500 de HellaSwag y 500 de ARC-Easy) la comparativa es
# estable (error estandar ~2 puntos) y el notebook entero cabe en clase.
# MMLU machine_learning tiene 112 preguntas: el limite no le afecta. None = benchmark completo (A100/H100).
LIMIT = 500

Compute dtype: torch.float16
Base: Qwen/Qwen3-4B-Base | Instruct: Qwen/Qwen3-4B


In [7]:
import lm_eval
from lm_eval.tasks import TaskManager
from lm_eval.models.huggingface import HFLM
import pandas as pd
import gc

task_manager = TaskManager()


def load_model(model_id):
    """Carga modelo + tokenizer en 4-bit."""
    print(f"Loading {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        dtype=COMPUTE_DTYPE,
        device_map="auto",
    )
    # Qwen trae max_new_tokens=2048 en su generation_config, y eso PISA el limite de generacion
    # que fija lm-eval (max_gen_toks). Lo quitamos para que mande lm-eval.
    model.generation_config.max_new_tokens = None
    print(f"  VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    return model, tokenizer


def free_model():
    """Libera VRAM antes de cargar el siguiente modelo.

    OJO: hay que hacer `del model_base` FUERA de la funcion. Un `del model` dentro de la funcion
    solo borra la variable local: la global sigue apuntando al modelo y la VRAM no se libera.
    """
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"  VRAM tras liberar: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


def eval_basic(model):
    """HellaSwag + ARC-Easy · 0-shot."""
    return lm_eval.simple_evaluate(
        model=HFLM(model, batch_size=8),
        tasks=TASKS_BASIC,
        task_manager=task_manager,
        num_fewshot=0,
        batch_size=8,
        limit=LIMIT,
    )


def eval_mmlu(model, num_fewshot):
    """MMLU machine_learning subset · n-shot loglikelihood."""
    return lm_eval.simple_evaluate(
        model=HFLM(model, batch_size=8),
        tasks=[TASK_MMLU],
        task_manager=task_manager,
        num_fewshot=num_fewshot,
        batch_size=8,
        limit=LIMIT,
    )


def eval_mmlu_cot(model):
    """MMLU machine_learning CoT 0-shot.

    Tarea generativa: el modelo escribe el razonamiento y luego "The answer is (X)".
    Limitamos a 384 tokens por respuesta (gen_kwargs). Sin limite, el modelo base no para de escribir
    y en una T4 la CoT sola tarda mas de media hora por modelo.
    Ojo: esta variante de lm-eval usa el split de validacion de MMLU-ML (11 preguntas): es una DEMO,
    mira el stderr antes de sacar conclusiones.
    """
    return lm_eval.simple_evaluate(
        model=HFLM(model, batch_size=4),
        tasks=[TASK_MMLU_COT],
        task_manager=task_manager,
        num_fewshot=0,
        batch_size=4,
        limit=LIMIT,
        gen_kwargs={"max_gen_toks": 384},
    )


def extract(results, task, metric="acc,none"):
    """Extrae una metrica concreta del dict de resultados."""
    return results["results"][task].get(metric, None)


print("Helpers definidos. Vamos.")

Helpers definidos. Vamos.


---

# Parte A · Modelo BASE (`MODEL_ID_BASE`)

Un modelo base **no** ha pasado por el post-entrenamiento (SFT + preferencias/RL). Predice el siguiente token y ya.

La hipotesis clasica (la que se cumplia con los base de 2020-2023):

- Le cuesta el formato "responde A/B/C/D".
- Le cuesta la CoT (no sabe generar razonamientos estructurados · solo continua texto).
- Va bien en tareas de pura continuacion (HellaSwag normalizado).

Ojo: los base de 2025-2026 (como `Qwen3-4B-Base`) se pre-entrenan con muchisimos datos de razonamiento e instrucciones. **Vamos a medir si la hipotesis sigue en pie.**


In [8]:
model_base, tokenizer_base = load_model(MODEL_ID_BASE)   # MODEL_ID_BASE viene del selector de arriba

Loading Qwen/Qwen3-4B-Base...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

  VRAM: 2.49 GB


### A.1 · HellaSwag + ARC-Easy (0-shot)

Estos 2 son los benchmarks ligeros del antiguo Open LLM Leaderboard (v1). Con `LIMIT = 500`: ~3-4 min en T4 en 4-bit.


In [ ]:
results_base_basic = eval_basic(model_base)
pd.DataFrame(results_base_basic["results"])

README.md:   0%|          | 0.00/7.02k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 24.4MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.11MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Running loglikelihood requests:  99%|█████████▉| 3970/3998 [01:42<00:00, 81.57it/s]

Running loglikelihood requests: 100%|█████████▉| 3979/3998 [01:42<00:00, 78.50it/s]

Running loglikelihood requests: 100%|█████████▉| 3992/3998 [01:42<00:00, 88.41it/s]

Running loglikelihood requests: 100%|██████████| 3998/3998 [01:42<00:00, 38.88it/s]

,hellaswag,arc_easy
name,hellaswag,arc_easy
alias,hellaswag,arc_easy
sample_len,500,500
"acc,none",0.506,0.746
"acc_stderr,none",0.022381,0.019487
"acc_norm,none",0.62,0.736
"acc_norm_stderr,none",0.021729,0.019733


### A.2 · MMLU machine_learning (5-shot y 0-shot)

MMLU completo son 57 dominios. Usamos solo el subset `machine_learning` para que dure razonablemente en clase.

**¿Por que 5-shot y 0-shot?**
- **5-shot** · el setting "oficial" de MMLU. El modelo ve 5 ejemplos antes de cada pregunta.
- **0-shot** · sin ejemplos. Segun la hipotesis clasica, el base sufre aqui porque no sabe "seguir instrucciones".

La diferencia entre ambos te dice cuanto depende de few-shot para responder.


In [ ]:
results_base_mmlu_5shot = eval_mmlu(model_base, num_fewshot=5)
pd.DataFrame(results_base_mmlu_5shot["results"])

README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

machine_learning/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 19.7kB            

machine_learning/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

machine_learning/validation-00000-of-000(…): reconstructing file:   0%|          |  0.00B / 6.17kB            

...

Running loglikelihood requests:  95%|█████████▍| 425/448 [03:52<00:11,  2.00it/s]

Running loglikelihood requests:  97%|█████████▋| 433/448 [03:56<00:07,  2.00it/s]

Running loglikelihood requests:  98%|█████████▊| 441/448 [04:00<00:03,  2.01it/s]

Running loglikelihood requests: 100%|██████████| 448/448 [04:00<00:00,  1.86it/s]

,mmlu_flan_n_shot_loglikelihood_machine_learning
name,mmlu_flan_n_shot_loglikelihood_machine_learning
alias,mmlu_flan_n_shot_loglikelihood_machine_learning
sample_len,112
"acc,none",0.651786
"acc_stderr,none",0.045218


In [ ]:
results_base_mmlu_0shot = eval_mmlu(model_base, num_fewshot=0)
pd.DataFrame(results_base_mmlu_0shot["results"])

  0%|          | 0/112 [00:00<?, ?it/s]

 56%|█████▋    | 63/112 [00:00<00:00, 619.76it/s]

100%|██████████| 112/112 [00:00<00:00, 612.89it/s]

Tokenizing inputs:   0%|          | 0/448 [00:00<?, ?it/s]

Tokenizing inputs:  45%|████▍     | 200/448 [00:00<00:00, 1998.28it/s]

...

Running loglikelihood requests:  95%|█████████▍| 425/448 [00:29<00:00, 23.37it/s]

Running loglikelihood requests:  97%|█████████▋| 433/448 [00:29<00:00, 23.82it/s]

Running loglikelihood requests:  98%|█████████▊| 441/448 [00:30<00:00, 24.39it/s]

Running loglikelihood requests: 100%|██████████| 448/448 [00:30<00:00, 14.78it/s]

,mmlu_flan_n_shot_loglikelihood_machine_learning
name,mmlu_flan_n_shot_loglikelihood_machine_learning
alias,mmlu_flan_n_shot_loglikelihood_machine_learning
sample_len,112
"acc,none",0.535714
"acc_stderr,none",0.047337


### A.3 · MMLU machine_learning · Chain of Thought (CoT)

CoT pide al modelo que **genere el razonamiento** antes de la respuesta ("Let's think step by step" ... "The answer is (X)"). La hipotesis clasica dice que un base lo hace mal porque no esta entrenado para producir ese texto estructurado.

Dos avisos de protocolo antes de mirar el numero:
- Esta tarea de lm-eval usa el split de validacion de MMLU-ML: **solo 11 preguntas**. Mira el `stderr` (~0,12-0,15): una pregunta arriba o abajo mueve 9 puntos.
- `flexible-extract` busca cualquier "(X)" en la respuesta; `strict-match` exige la frase "The answer is". Comparar las dos te dice si el modelo razona bien pero no respeta el formato.


In [ ]:
results_base_mmlu_cot = eval_mmlu_cot(model_base)
pd.DataFrame(results_base_mmlu_cot["results"])

  0%|          | 0/11 [00:00<?, ?it/s]

100%|██████████| 11/11 [00:00<00:00, 408.79it/s]

Running generate_until requests:   0%|          | 0/11 [00:00<?, ?it/s]

Running generate_until requests:   9%|▉         | 1/11 [00:33<05:30, 33.03s/it]

Running generate_until requests:  45%|████▌     | 5/11 [01:05<01:10, 11.74s/it]

Running generate_until requests:  82%|████████▏ | 9/11 [01:37<00:19,  9.79s/it]

Running generate_until requests: 100%|██████████| 11/11 [01:37<00:00,  8.90s/it]

,mmlu_flan_cot_zeroshot_machine_learning
name,mmlu_flan_cot_zeroshot_machine_learning
alias,mmlu_flan_cot_zeroshot_machine_learning
sample_len,11
"exact_match,strict-match",0.181818
"exact_match_stderr,strict-match",0.121967
"exact_match,flexible-extract",0.818182
"exact_match_stderr,flexible-extract",0.121967


### A.4 · Liberar memoria

Antes de cargar el instruct, soltamos el base. Si no, los dos modelos conviven en la GPU (en 4-bit caben en la T4, pero con modelos mayores te quedas sin VRAM). Deberias ver la VRAM volver a ~0 GB.


In [ ]:
del model_base      # borra la referencia global (sin esto la VRAM NO se libera)
free_model()

  VRAM tras liberar: 0.01 GB


---

# Parte B · Modelo INSTRUCT (`MODEL_ID_INSTRUCT`)

`Qwen/Qwen3-4B` es el mismo modelo tras el post-entrenamiento (SFT + RL, con modo de razonamiento "thinking" integrado). La hipotesis clasica espera:

- Mejor en formato A/B/C/D (sabe seguir la instruccion "elige una opcion").
- Mejor en CoT (sabe generar razonamiento estructurado).
- HellaSwag/ARC-Easy parecidos al base (miden conocimiento del mundo; el post-entrenamiento no anade conocimiento).

Ejecutamos los mismos 4 benchmarks, **con el mismo protocolo** (prompts crudos, sin chat template, 4-bit).


In [ ]:
model_instruct, tokenizer_instruct = load_model(MODEL_ID_INSTRUCT)   # MODEL_ID_INSTRUCT viene del selector

Loading Qwen/Qwen3-4B...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  VRAM: 2.50 GB


In [ ]:
results_instruct_basic = eval_basic(model_instruct)
pd.DataFrame(results_instruct_basic["results"])

  0%|          | 0/500 [00:00<?, ?it/s]

 25%|██▌       | 126/500 [00:00<00:00, 1257.22it/s]

 50%|█████     | 252/500 [00:00<00:00, 1222.96it/s]

 75%|███████▌  | 375/500 [00:00<00:00, 1218.75it/s]

 99%|█████████▉| 497/500 [00:00<00:00, 1216.59it/s]

...

Running loglikelihood requests:  99%|█████████▉| 3970/3998 [01:56<00:00, 89.84it/s]

Running loglikelihood requests: 100%|█████████▉| 3979/3998 [01:56<00:00, 83.53it/s]

Running loglikelihood requests: 100%|█████████▉| 3992/3998 [01:56<00:00, 92.06it/s]

Running loglikelihood requests: 100%|██████████| 3998/3998 [01:56<00:00, 34.18it/s]

,hellaswag,arc_easy
name,hellaswag,arc_easy
alias,hellaswag,arc_easy
sample_len,500,500
"acc,none",0.476,0.786
"acc_stderr,none",0.022357,0.01836
"acc_norm,none",0.57,0.766
"acc_norm_stderr,none",0.022163,0.018953


In [ ]:
results_instruct_mmlu_5shot = eval_mmlu(model_instruct, num_fewshot=5)
pd.DataFrame(results_instruct_mmlu_5shot["results"])

  0%|          | 0/112 [00:00<?, ?it/s]

 12%|█▏        | 13/112 [00:00<00:00, 127.46it/s]

 23%|██▎       | 26/112 [00:00<00:00, 127.00it/s]

 36%|███▌      | 40/112 [00:00<00:00, 128.54it/s]

 47%|████▋     | 53/112 [00:00<00:00, 122.94it/s]

...

Running loglikelihood requests:  95%|█████████▍| 425/448 [03:53<00:11,  2.01it/s]

Running loglikelihood requests:  97%|█████████▋| 433/448 [03:57<00:07,  2.01it/s]

Running loglikelihood requests:  98%|█████████▊| 441/448 [04:01<00:03,  2.01it/s]

Running loglikelihood requests: 100%|██████████| 448/448 [04:01<00:00,  1.86it/s]

,mmlu_flan_n_shot_loglikelihood_machine_learning
name,mmlu_flan_n_shot_loglikelihood_machine_learning
alias,mmlu_flan_n_shot_loglikelihood_machine_learning
sample_len,112
"acc,none",0.491071
"acc_stderr,none",0.04745


In [ ]:
results_instruct_mmlu_0shot = eval_mmlu(model_instruct, num_fewshot=0)
pd.DataFrame(results_instruct_mmlu_0shot["results"])

  0%|          | 0/112 [00:00<?, ?it/s]

 54%|█████▎    | 60/112 [00:00<00:00, 598.64it/s]

100%|██████████| 112/112 [00:00<00:00, 603.19it/s]

Tokenizing inputs:   0%|          | 0/448 [00:00<?, ?it/s]

Tokenizing inputs:  43%|████▎     | 194/448 [00:00<00:00, 1938.10it/s]

...

Running loglikelihood requests:  95%|█████████▍| 425/448 [00:29<00:00, 23.20it/s]

Running loglikelihood requests:  97%|█████████▋| 433/448 [00:30<00:00, 23.71it/s]

Running loglikelihood requests:  98%|█████████▊| 441/448 [00:30<00:00, 24.31it/s]

Running loglikelihood requests: 100%|██████████| 448/448 [00:30<00:00, 14.76it/s]

,mmlu_flan_n_shot_loglikelihood_machine_learning
name,mmlu_flan_n_shot_loglikelihood_machine_learning
alias,mmlu_flan_n_shot_loglikelihood_machine_learning
sample_len,112
"acc,none",0.526786
"acc_stderr,none",0.04739


In [ ]:
results_instruct_mmlu_cot = eval_mmlu_cot(model_instruct)
pd.DataFrame(results_instruct_mmlu_cot["results"])

  0%|          | 0/11 [00:00<?, ?it/s]

100%|██████████| 11/11 [00:00<00:00, 409.39it/s]

Running generate_until requests:   0%|          | 0/11 [00:00<?, ?it/s]

Running generate_until requests:   9%|▉         | 1/11 [00:32<05:25, 32.50s/it]

Running generate_until requests:  45%|████▌     | 5/11 [01:04<01:10, 11.72s/it]

Running generate_until requests:  82%|████████▏ | 9/11 [01:37<00:19,  9.72s/it]

Running generate_until requests: 100%|██████████| 11/11 [01:37<00:00,  8.83s/it]

,mmlu_flan_cot_zeroshot_machine_learning
name,mmlu_flan_cot_zeroshot_machine_learning
alias,mmlu_flan_cot_zeroshot_machine_learning
sample_len,11
"exact_match,strict-match",0.181818
"exact_match_stderr,strict-match",0.121967
"exact_match,flexible-extract",0.363636
"exact_match_stderr,flexible-extract",0.15212


---

# Cierre · Comparativa base vs instruct

La pregunta del modulo: **¿cuanto sube por instruct tuning?** (spoiler: depende de COMO lo midas).

Construimos una tabla que junta los benchmarks de ambos modelos (MMLU-ML en 5-shot, 0-shot y CoT) + el delta.

> Nota tecnica sobre HellaSwag · usamos `acc_norm` (length-normalized accuracy), no `acc`. La razon: HellaSwag es multiple-choice con opciones de longitud variable, y un modelo tiende a preferir respuestas mas cortas si no normalizas por longitud. **`acc_norm` es la metrica canonica** que se reporta en el Open LLM Leaderboard.


In [ ]:
def safe_extract(results, task, metric):
    """Extrae metrica, devuelve None si no existe."""
    try:
        return results["results"][task].get(metric)
    except (KeyError, TypeError):
        return None

filas = [
    {
        "Benchmark": "HellaSwag (acc_norm)",
        "base": safe_extract(results_base_basic, "hellaswag", "acc_norm,none"),
        "instruct": safe_extract(results_instruct_basic, "hellaswag", "acc_norm,none"),
    },
    {
        "Benchmark": "ARC-Easy (acc)",
        "base": safe_extract(results_base_basic, "arc_easy", "acc,none"),
        "instruct": safe_extract(results_instruct_basic, "arc_easy", "acc,none"),
    },
    {
        "Benchmark": "MMLU-ML 5-shot (acc)",
        "base": safe_extract(results_base_mmlu_5shot, TASK_MMLU, "acc,none"),
        "instruct": safe_extract(results_instruct_mmlu_5shot, TASK_MMLU, "acc,none"),
    },
    {
        "Benchmark": "MMLU-ML 0-shot (acc)",
        "base": safe_extract(results_base_mmlu_0shot, TASK_MMLU, "acc,none"),
        "instruct": safe_extract(results_instruct_mmlu_0shot, TASK_MMLU, "acc,none"),
    },
    {
        "Benchmark": "MMLU-ML CoT (exact_match)",
        "base": safe_extract(results_base_mmlu_cot, TASK_MMLU_COT, "exact_match,flexible-extract"),
        "instruct": safe_extract(results_instruct_mmlu_cot, TASK_MMLU_COT, "exact_match,flexible-extract"),
    },
]

df = pd.DataFrame(filas)
df["delta_pp"] = (df["instruct"] - df["base"]).round(3)
df["delta_%"] = ((df["instruct"] - df["base"]) / df["base"] * 100).round(1)

df

,Benchmark,base,instruct,delta_pp,delta_%
0,HellaSwag (acc_norm),0.620000,0.570000,-0.050,-8.1
1,ARC-Easy (acc),0.746000,0.786000,0.040,5.4
2,MMLU-ML 5-shot (acc),0.651786,0.491071,-0.161,-24.7
3,MMLU-ML 0-shot (acc),0.535714,0.526786,-0.009,-1.7
4,MMLU-ML CoT (exact_match),0.818182,0.363636,-0.455,-55.6


## Interpretacion · que dicen los numeros

Ejecucion de referencia (oct-2026 · T4 · 4-bit NF4 con compute fp16 · `LIMIT = 500`). Tus numeros pueden moverse unas centesimas:

| Benchmark | base | instruct | Lectura |
|---|---|---|---|
| HellaSwag (acc_norm, n=500) | 0.62 | 0.57 | Baja ligeramente · el post-entrenamiento no anade conocimiento y desplaza al modelo de la "continuacion de texto" |
| ARC-Easy (acc, n=500) | 0.75 | 0.79 | Sube un poco (stderr ~0.02) |
| MMLU-ML 5-shot (n=112) | 0.65 | 0.49 | **Baja** · el instruct evaluado sin su chat template pierde calibracion en el formato "(A)/(B)/..." |
| MMLU-ML 0-shot (n=112) | 0.54 | 0.53 | Igual (stderr ~0.05) |
| MMLU-ML CoT (flexible, n=11) | 0.82 | 0.36 | El base razona muy bien; el instruct cae. Con 11 preguntas, es una señal, no una prueba |

**La hipotesis clasica NO se cumple aqui, y esa es la leccion:**

1. **Los base de 2025-2026 ya no son "tontos".** `Qwen3-4B-Base` se pre-entrena con mucho texto de razonamiento e instrucciones: sabe hacer CoT y responder A/B/C/D sin post-entrenamiento.
2. **El protocolo manda.** Medimos el instruct con prompts crudos y sin chat template, que no es como se usa. Un modelo post-entrenado (y con modo thinking) fuera de su formato suele puntuar peor: es una de las razones por las que el mismo modelo saca cifras distintas en distintos leaderboards. En lm-eval puedes repetirlo con `apply_chat_template=True` en `simple_evaluate`.
3. **Muestras pequenas = ruido.** CoT con 11 preguntas tiene un stderr de ~0.13; antes de concluir, mira el error estandar.

**Conclusion practica** · no elijas modelo por un numero suelto de benchmark: compara siempre con el MISMO protocolo, el adecuado para cada modelo (chat template para los instruct), y con muestras suficientes. Y para tu caso de uso, la eval que manda es la tuya (módulo de evaluación de LLM apps, clase 6).

## Comparativa con frontier 2026

Tus numeros en MMLU machine_learning vs la frontera de octubre 2026 (slide 20):

- Frontier por API (Gemini 3 Pro, Claude Opus 4.5) · MMLU-Pro ~0.90 (techo actual)
- Gemma 4 E4B (local, cabe en T4) · MMLU-Pro 0.694 (model card oficial de Google)
- Qwen3-4B (este notebook) · tu MMLU-ML de arriba (ojo: MMLU clasico, no MMLU-Pro; no son directamente comparables)

El gap con el frontier es **real** para un modelo de 4B (~20 puntos de MMLU-Pro). Pero acordate de las slides 20 y 26 · **MMLU clasico esta saturado**. Para diferenciar frontier hoy se mira MMLU-Pro, GPQA Diamond, SWE-bench Pro o HLE.

## Siguiente paso · evalua tu modelo fine-tuneado

Repite este notebook con tus adapters de Fine-tuning: el del lab 04 (Milei, QLoRA sobre Qwen3.5) o el del lab 05 (distillation, Gemma 4 E2B). `HFLM` acepta el modelo con el adapter cargado (`PeftModel.from_pretrained(base, "tu_usuario/milei-qlora-adapter")`), igual que en el lab 09 de Fine-tuning. Lectura: estos benchmarks generales miden **retencion**. Un fine-tune de estilo o de dominio no deberia subir HellaSwag; lo que buscas es que **no baje**. Lo que tu fine-tune gana (estilo, tu tarea) se mide con una eval propia, como hiciste en el lab 09 de Fine-tuning.

Eso es **el cierre real del flow Fine-tuning → Evaluación**.

## Lighteval · la alternativa moderna (mencion)

`lighteval` de HuggingFace cubre el mismo proposito que `lm-evaluation-harness` con API mas integrada al ecosistema HF. Sintaxis tipica:

```bash
pip install lighteval[accelerate]
lighteval accelerate \
    --model_args "pretrained=Qwen/Qwen3-4B,load_in_4bit=True" \
    --tasks "leaderboard|hellaswag|0|0,leaderboard|arc:easy|0|0" \
    --output_dir ./results
```

Misma idea, distinto wrapper. Si trabajas en HuggingFace stack, lighteval encaja mejor.
